In [1]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [14]:
Epilepsy_Cohort_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cleaned_Lab_Cohort_toStack")

In [3]:
Epilepsy_Cohort_Lab.printSchema()

▸,:,


root
 |-- labcode: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



In [15]:
Epilepsy_Cohort_Lab.show(truncate=False)

+-------+------------------------------------+--------------------------+-----------+
|labcode|personid                            |New_updated_Interpretation|servicedate|
+-------+------------------------------------+--------------------------+-----------+
|13303-3|52ad17e0-58fc-4b53-ade5-1684cd0808d9|Normal                    |2022-04-08 |
|15283-5|89f8f307-08b9-4925-b42f-2ef3690b943d|null                      |2017-06-15 |
|15283-5|50d97b3b-0381-4460-a227-7b3ee82d6a95|Normal                    |2017-07-25 |
|15283-5|2711acb4-15e9-49be-b2d8-cf6b8c529ad0|Normal                    |2017-06-28 |
|15283-5|6cc4b0ff-5181-435e-8b53-6e6a1403c1a4|null                      |2019-10-18 |
|15283-5|aedd9a41-3877-4cf0-93a5-9c5dbc146362|High                      |2017-05-21 |
|15283-5|73463313-2c66-47ff-bb36-62f8058e80d0|null                      |2021-06-02 |
|15283-5|984922c4-80b7-4656-966e-116c7bf72313|Normal                    |2018-05-10 |
|15283-5|58631d07-24d3-427e-b7ac-e3231cff377c|null    

In [18]:
from pyspark.sql.functions import col, count, when
# Calculate the percentage of null New_updated_Interpretation per labcode
result = Epilepsy_Cohort_Lab.groupBy("labcode").agg(
    (count(when(col("New_updated_Interpretation").isNull(), 1)) / count("*")).alias("null_percentage")
)

# Filter labcodes having more than 90% null New_updated_Interpretation
filtered_result = result.filter(col("null_percentage") > 0.9)

# Count the unique labcodes
unique_labcodes_count = filtered_result.count()

# Show the result
print(f"Count of unique labcodes with more than 90% null New_updated_Interpretation: {unique_labcodes_count}")

Count of unique labcodes with more than 90% null New_updated_Interpretation: 0


In [19]:
from pyspark.sql.functions import countDistinct
# Group by labcode and personid, then count distinct New_updated_Interpretation
result_df = Epilepsy_Cohort_Lab.groupBy("labcode", "personid") \
    .agg(countDistinct("New_updated_Interpretation").alias("interpretation_count")) \
    .filter("interpretation_count > 1")

# Show the result
result_df.show(truncate=False)

+-------+--------+--------------------+
|labcode|personid|interpretation_count|
+-------+--------+--------------------+
+-------+--------+--------------------+



In [14]:
Epilepsy_Control_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cleaned_Lab_Control_toStack")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
from pyspark.sql.functions import countDistinct
# Group by labcode and personid, then count distinct New_updated_Interpretation
result_df1 = Epilepsy_Control_Lab.groupBy("labcode", "personid") \
    .agg(countDistinct("New_updated_Interpretation").alias("interpretation_count")) \
    .filter("interpretation_count > 1")

# Show the result
result_df1.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-------+--------+--------------------+
|labcode|personid|interpretation_count|
+-------+--------+--------------------+
+-------+--------+--------------------+



<IPython.core.display.Javascript object>

In [3]:
# Count the unique combinations of personid and labcode
unique_combinations_count = Epilepsy_Control_Lab \
    .select("personid", "labcode") \
    .distinct() \
    .count()

# Display the result
print(f"Total unique combinations of personid and labcode: {unique_combinations_count}")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Total unique combinations of personid and labcode: 15782777


<IPython.core.display.Javascript object>

In [6]:
# Filter the DataFrame for the specified personid
filtered_df = Epilepsy_Control_Lab \
    .filter(Epilepsy_Control_Lab.personid == "01ab70e7-d27c-4dad-8e10-b1116b6b1fe6")

# Display the filtered records
filtered_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-------+------------------------------------+--------------------------+-----------+
|labcode|personid                            |New_updated_Interpretation|servicedate|
+-------+------------------------------------+--------------------------+-----------+
|2160-0 |01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|Normal                    |2021-10-05 |
|751-8  |01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|High                      |2021-11-23 |
|5905-5 |01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|Normal                    |2021-11-23 |
|718-7  |01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|High                      |2021-11-23 |
|43305-2|01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|null                      |2021-02-23 |
|4544-3 |01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|High                      |2021-11-23 |
|5803-2 |01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|Normal                    |2021-10-03 |
|2885-2 |01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|Normal                    |2021-10-05 |
|711-2  |01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|Normal  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# Filter the DataFrame for non-null values in New_updated_Interpretation
non_null_interpretation_df = filtered_df \
    .filter(filtered_df.New_updated_Interpretation.isNotNull())

# Display the filtered records
non_null_interpretation_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-------+------------------------------------+--------------------------+-----------+
|labcode|personid                            |New_updated_Interpretation|servicedate|
+-------+------------------------------------+--------------------------+-----------+
|2160-0 |01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|Normal                    |2021-10-05 |
|751-8  |01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|High                      |2021-11-23 |
|5905-5 |01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|Normal                    |2021-11-23 |
|718-7  |01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|High                      |2021-11-23 |
|4544-3 |01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|High                      |2021-11-23 |
|5803-2 |01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|Normal                    |2021-10-03 |
|2885-2 |01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|Normal                    |2021-10-05 |
|711-2  |01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|Normal                    |2021-11-23 |
|17861-6|01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|Normal  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
print(non_null_interpretation_df.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

66


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
from pyspark.sql.functions import countDistinct

# Group by personid and count distinct labcodes for each personid
personid_labcodes_count = Epilepsy_Control_Lab \
    .groupBy("personid") \
    .agg(countDistinct("labcode").alias("unique_labcode_count"))

# Display the result
personid_labcodes_count.show(truncate=False)

+------------------------------------+--------------------+
|personid                            |unique_labcode_count|
+------------------------------------+--------------------+
|01ab70e7-d27c-4dad-8e10-b1116b6b1fe6|113                 |
|15a4e0c3-3666-41e6-adf5-5c7b87f0c57b|67                  |
|176c028f-02d9-4471-b7a8-398c3c943239|78                  |
|1c0a0da6-cd89-4953-a569-2c0e4c6dcae5|57                  |
|1f2e5f9d-cfb3-4a42-bd21-9c2734255a2a|40                  |
|2b89e83d-f7d1-4ace-b07c-2b524bf488b8|25                  |
|306271aa-b715-4346-a125-502f82f7f4ff|89                  |
|311317e0-27dd-40ab-bfba-68664fb82038|54                  |
|31bbc7c1-ec1e-4055-a169-cd3977033146|65                  |
|34c6f283-20c1-4951-bcf3-ce7824ef47b6|87                  |
|399cb86e-504f-42c5-ad2f-4d9e0a299a91|38                  |
|3b0e4f86-62d0-40bf-a8d7-12955f3dc93a|44                  |
|4197bbc0-c06b-403f-9fa7-ee41bf4474cb|60                  |
|41df2b89-5fc7-442a-b8c6-80c57e9b5ff3|26

In [16]:
Epilepsy_Control_Pivoted = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya_control_lab_pivot")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
# Get the list of column names
column_names = Epilepsy_Control_Pivoted.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns Cohort:", num_columns)

▸,:,


Number of columns Cohort: 5966


In [19]:
# Count the number of unique lab codes
unique_labcodes_count = Epilepsy_Control_Lab.select("labcode").distinct().count()

# Display the number of unique lab codes
print("Number of unique lab codes:", unique_labcodes_count)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Number of unique lab codes: 5974


<IPython.core.display.Javascript object>

In [7]:
Epilepsy_Control_Pivoted.printSchema()

root
 |-- personid: string (nullable = true)
 |-- 1004-1: string (nullable = true)
 |-- 1005-8: string (nullable = true)
 |-- 1006-6: string (nullable = true)
 |-- 1007-4: string (nullable = true)
 |-- 10328-3: string (nullable = true)
 |-- 10329-1: string (nullable = true)
 |-- 10331-7: string (nullable = true)
 |-- 10332-5: string (nullable = true)
 |-- 10333-3: string (nullable = true)
 |-- 10334-1: string (nullable = true)
 |-- 10335-8: string (nullable = true)
 |-- 10338-2: string (nullable = true)
 |-- 1034-8: string (nullable = true)
 |-- 10341-6: string (nullable = true)
 |-- 10346-5: string (nullable = true)
 |-- 10350-7: string (nullable = true)
 |-- 10352-3: string (nullable = true)
 |-- 10354-9: string (nullable = true)
 |-- 10360-6: string (nullable = true)
 |-- 10362-2: string (nullable = true)
 |-- 10365-5: string (nullable = true)
 |-- 10368-9: string (nullable = true)
 |-- 10371-3: string (nullable = true)
 |-- 10373-9: string (nullable = true)
 |-- 10374-7: string (nu

In [2]:
# Filter the DataFrame for the specified personid
filtered_df = Epilepsy_Control_Pivoted \
    .filter(Epilepsy_Control_Pivoted.personid == "01ab70e7-d27c-4dad-8e10-b1116b6b1fe6")

# Display the filtered records
filtered_df.show(truncate=False)

▸,:,


NameError: name 'Epilepsy_Control_Pivoted' is not defined

In [12]:
# Select the specific columns
selected_columns_df = filtered_df.select("2160-0", "751-8", "5905-5", "718-7")

# Display the selected columns
selected_columns_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------+-----+------+-----+
|2160-0|751-8|5905-5|718-7|
+------+-----+------+-----+
|Normal|High |Normal|High |
+------+-----+------+-----+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
# Drop rows with null values in any column
filtered_df_no_nulls = filtered_df.dropna()

# # Select the specific columns
# selected_columns_df = filtered_df_no_nulls.select("2160-0", "751-8", "5905-5", "718-7")

# Display the selected columns
filtered_df_no_nulls.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------+------+------+------+------+-------+-------+-------+-------+-------+-------+-------+-------+------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+---

<IPython.core.display.Javascript object>

In [7]:
Epilepsy_Cohort_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya-bigjoin")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
Epilepsy_Control_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya_control_lab_pivot")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
print(Epilepsy_Cohort_Lab.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

118524


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
print(Epilepsy_Control_Lab.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

412634


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# Get the list of column names
column_names = Epilepsy_Control_Lab.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

▸,:,


Number of columns: 5966


### Test - Cohort+Control - For Count of Nulls - DataCheck

In [28]:
Lab1 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya-labCohortFinalRPN.parquet")

In [19]:
# Get the list of column names
column_names = Lab1.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

Number of columns: 4707


In [1]:
# Epilepsy_Cohort_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_pivoted_Cohort_Lab2")
Epilepsy_Cohort_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cleaned_Lab_Cohort_toStack")

In [5]:
Epilepsy_Cohort_Lab.printSchema()

▸,:,


root
 |-- labcode: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



In [12]:
from pyspark.sql.functions import countDistinct, col
# Step 1: Calculate the total count of unique personids
unique_personid_count = Epilepsy_Cohort_Lab.select(countDistinct("personid").alias("unique_personid_count")).collect()[0]["unique_personid_count"]

# Step 2: Calculate the count of distinct personids for each labcode
personid_counts_per_labcode = Epilepsy_Cohort_Lab.groupBy("labcode").agg(countDistinct("personid").alias("personid_count"))

# Step 3: Filter the labcodes having personid count less than 90% of the unique personid count
threshold = unique_personid_count * 0.9
filtered_labcodes = personid_counts_per_labcode.filter(col("personid_count") < threshold)

# Step 4: Count the unique labcodes that satisfy the condition
unique_labcodes_count = filtered_labcodes.select(countDistinct("labcode").alias("unique_labcodes_count"))

# Show the result
unique_labcodes_count.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+---------------------+
|unique_labcodes_count|
+---------------------+
|5321                 |
+---------------------+



<IPython.core.display.Javascript object>

In [13]:
from pyspark.sql.functions import countDistinct

# Group by labcode and count the number of distinct personid for each labcode
personid_counts_per_labcode = Epilepsy_Cohort_Lab.groupBy("labcode").agg(countDistinct("personid").alias("personid_count"))

# Show the result
personid_counts_per_labcode.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-------+--------------+
|labcode|personid_count|
+-------+--------------+
|11274-8|836           |
|11556-8|1291          |
|23826-1|729           |
|29541-0|166           |
|3043-7 |88            |
|33411-0|297           |
|50224-5|396           |
|703-9  |355           |
|16823-7|11            |
|20575-7|40            |
|40987-0|455           |
|74736-0|91            |
|32294-1|146           |
|68916-6|10            |
|21397-5|7             |
|21419-7|10            |
|81285-9|3             |
|89349-5|3             |
|4547-6 |46            |
|6007-9 |36            |
+-------+--------------+
only showing top 20 rows



<IPython.core.display.Javascript object>

In [18]:
from pyspark.sql.functions import col, max
# Find the maximum personid_count
max_personid_count = personid_counts_per_labcode.select(max(col("personid_count"))).collect()[0][0]

# Display the result
print(f"The highest personid_count is: {max_personid_count}")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The highest personid_count is: 93584


<IPython.core.display.Javascript object>

In [15]:
from pyspark.sql.functions import col

# Calculate the count of distinct personids for each labcode
personid_counts_per_labcode = Epilepsy_Cohort_Lab.groupBy("labcode").agg(countDistinct("personid").alias("personid_count"))

# Filter the labcodes having personid_count greater than 1000
filtered_labcodes = personid_counts_per_labcode.filter(col("personid_count") > 1000)

# Count the number of labcodes that satisfy the condition
count_labcodes = filtered_labcodes.count()

# Show the result
print(f"Total number of labcodes with personid_count > 1000: {count_labcodes}")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Total number of labcodes with personid_count > 1000: 490


<IPython.core.display.Javascript object>

In [10]:
from pyspark.sql.functions import count, countDistinct, col

# Step 1: Calculate the total count of unique labcodes
unique_labcodes_count = Epilepsy_Cohort_Lab.select(countDistinct("labcode").alias("unique_labcodes_count")).collect()[0]["unique_labcodes_count"]

# Step 2: Calculate the labcode count for each personid
labcode_counts = Epilepsy_Cohort_Lab.groupBy("personid").agg(count("labcode").alias("labcode_count"))

# Step 3: Filter the personids having labcode count less than 90% of the unique labcodes count
threshold = unique_labcodes_count * 0.9
filtered_personids = labcode_counts.filter(col("labcode_count") < threshold)

# Step 4: Count the unique personids that satisfy the condition
unique_personids_count = filtered_personids.select(countDistinct("personid").alias("unique_personids_count"))

# Show the result
unique_personids_count.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+----------------------+
|unique_personids_count|
+----------------------+
|118524                |
+----------------------+



<IPython.core.display.Javascript object>

In [2]:
from pyspark.sql.functions import count, countDistinct

# Group by personid and count the number of labcodes for each personid
labcode_counts = Epilepsy_Cohort_Lab.groupBy("personid").agg(count("labcode").alias("labcode_count"))

# Show the result
labcode_counts.show(truncate=False)
# Count the total number of unique labcodes
unique_labcodes_count = Epilepsy_Cohort_Lab.select(countDistinct("labcode").alias("unique_labcodes_count"))

# Show the result
unique_labcodes_count.show(truncate=False)
# Count the total number of unique labcodes
unique_personid_count = Epilepsy_Cohort_Lab.select(countDistinct("personid").alias("unique_personid_count"))
# # Show the result
unique_personid_count.show(truncate=False)

+------------------------------------+-------------+
|personid                            |labcode_count|
+------------------------------------+-------------+
|9eae9705-0e6f-47b4-a6b1-e4a165b4bab2|116          |
|e2cba0e4-aaf0-4a27-b376-781aacc4c182|88           |
|2ed351f8-d552-462b-9607-03a69ba65ec6|43           |
|9cc294bc-bb24-4cff-aa51-82433ed0e4b5|96           |
|51f089a9-218c-4fbd-85cf-fcb9ffcc2d75|35           |
|b55031c9-5d23-4377-9d46-d2ca47f3a467|81           |
|0b850d24-f3eb-495d-8928-a837a7609107|46           |
|48fcecc9-a86f-4263-9c63-766c2fac26c7|43           |
|7a0776e1-3d1f-4f77-b987-460314ae3c05|57           |
|bfb6e323-629b-4949-8e3b-d1965f252da1|24           |
|8947aa9d-b420-4401-b012-6debdf8ae0d8|47           |
|869c8fd8-9a08-4a4c-9cc2-1578129ed0b3|78           |
|79c48c09-c671-4c3e-ab60-739dab5b5468|125          |
|a117f286-0c98-4c52-bec1-fb81875879f3|33           |
|74f93285-ce4b-4ad1-91b8-d663f5cd3355|37           |
|b25bd905-13c9-4d00-a355-c64673cd5caf|56      

In [4]:
from pyspark.sql.functions import col

# Filter the DataFrame for the specific personid
result = labcode_counts.filter(col("personid") == "e502385e-fd5c-45bb-bbe0-f6b4aac743e2")

# Show the result
result.show(truncate=False)

+------------------------------------+-------------+
|personid                            |labcode_count|
+------------------------------------+-------------+
|e502385e-fd5c-45bb-bbe0-f6b4aac743e2|105          |
+------------------------------------+-------------+



In [3]:
from pyspark.sql.functions import col, max

# Find the maximum labcode_count
max_labcode_count = labcode_counts.agg(max("labcode_count").alias("max_labcode_count")).collect()[0]["max_labcode_count"]

# Filter to get the personid(s) with the maximum labcode_count
persons_with_max_count = labcode_counts.filter(col("labcode_count") == max_labcode_count)

# Show the result
persons_with_max_count.show(truncate=False)

+------------------------------------+-------------+
|personid                            |labcode_count|
+------------------------------------+-------------+
|b4ca96a2-cfec-4902-97de-ac07e857be47|275          |
+------------------------------------+-------------+



In [16]:
from pyspark.sql.functions import max

# Calculate the labcode count for each personid
labcode_counts = Epilepsy_Cohort_Lab.groupBy("personid").agg(count("labcode").alias("labcode_count"))

# Find the maximum labcode_count
max_labcode_count = labcode_counts.agg(max("labcode_count").alias("max_labcode_count")).collect()[0]["max_labcode_count"]

# Show the result
print(f"The highest labcode_count is: {max_labcode_count}")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The highest labcode_count is: 275


<IPython.core.display.Javascript object>

In [17]:
from pyspark.sql.functions import countDistinct, col

# Step 1: Filter the labcode_counts DataFrame to include only rows with labcode_count > 200
filtered_personids = labcode_counts.filter(col("labcode_count") > 200)

# Step 2: Count the distinct personids in the filtered DataFrame
unique_personids_count = filtered_personids.select(countDistinct("personid").alias("unique_personids_count"))

# Show the result
unique_personids_count.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+----------------------+
|unique_personids_count|
+----------------------+
|73                    |
+----------------------+



<IPython.core.display.Javascript object>

In [23]:
# Get the list of column names
column_names = Epilepsy_Cohort_Lab.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

Number of columns: 5322


In [3]:
Lab2 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya-labControlFinalRPN.parquet")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# Filter and display all column values for the specified personid
Lab1.filter(Lab1.personid == "2b3a15f5-22d5-4166-990c-3fc713b6a331").show(truncate=False)

+------------------------------------+-------+-------+-------+-------+-------+-------+-------+-------+------+-------+-------+-------+-------+------+-------+-------+-------+------+------+-------+-------+------+-------+-------+-------+------+-------+------+-------+-------+-------+------+------+-------+-------+-------+-------+-------+-------+-------+-----+-------+------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+------+-------+-------+-------+------+-------+------+-------+------+-------+-------+-------+------+-------+------+-------+------+-------+-------+-------+------+-------+-------+------+-------+------+-------+-------+-------+------+-----+-------+-------+-------+------+-------+-------+-------+------+-------+-------+-------+------+-------+-------+-------+------+------+-------+------+-------+-------+-------+------+-------+-------+-------+-------+-------+-------+------+------+-------+------+-------+-------+-------+------+------+------+-------+-------+

In [4]:
Lab1.printSchema()

root
 |-- personid: string (nullable = true)
 |-- 69726-8: string (nullable = true)
 |-- 40926-8: string (nullable = true)
 |-- 45142-7: string (nullable = true)
 |-- 32629-8: string (nullable = true)
 |-- 33768-3: string (nullable = true)
 |-- 17117-3: string (nullable = true)
 |-- 15284-3: string (nullable = true)
 |-- 38198-8: string (nullable = true)
 |-- 6035-0: string (nullable = true)
 |-- 46420-6: string (nullable = true)
 |-- 41647-9: string (nullable = true)
 |-- 62461-9: string (nullable = true)
 |-- 87621-9: string (nullable = true)
 |-- 1825-9: string (nullable = true)
 |-- 41215-5: string (nullable = true)
 |-- 29142-7: string (nullable = true)
 |-- 42679-1: string (nullable = true)
 |-- 787-2: string (nullable = true)
 |-- 1809-3: string (nullable = true)
 |-- 20640-9: string (nullable = true)
 |-- 73582-9: string (nullable = true)
 |-- 5777-8: string (nullable = true)
 |-- 82456-5: string (nullable = true)
 |-- 21203-5: string (nullable = true)
 |-- 21406-4: string (nul

In [5]:
totalCols = Lab1.columns
countCols = len(totalCols)
print("Total No of Columns:", countCols)

Total No of Columns: 4707


In [29]:
import pandas as pd
import math
from pyspark.sql.functions import col, lit, when, count
# Define the function to split the DataFrame
def split_df(df, num_split):
    total_columns = len(df.columns)
    first_column = df.columns[0]  # Get the name of the first column
    
    splitted = []
    
    num_columns_per_split = math.ceil((total_columns - 1) / num_split)  # Subtract 1 to exclude the first column
    
    for i in range(num_split):
        start = 1 + i * num_columns_per_split  # Start from the second column
        end = min(1 + (i + 1) * num_columns_per_split, total_columns)  # Add 1 to adjust for the first column
        split_columns = [first_column] + list(df.columns[start:end])
        split_df = df[split_columns]
        splitted.append(split_df)
        
    return splitted

In [30]:
# from pyspark.sql import functions as F
# from functools import reduce

# # Define the function to calculate the percentage of null values
# def find_columns_with_high_null_percentage(df, threshold=0.9, exclude_column=None):
#     """
#     Count the number of records having null values in percentage for each column
#     and display the total number of columns having more than 90% records with null values.

#     :param df: Input DataFrame
#     :param threshold: Threshold percentage for null values (default is 0.9 for 90%)
#     :param exclude_column: Column to exclude from the null check (default is None)
#     :return: Tuple with DataFrame of null percentages and count of columns with high null percentage
#     """
#     columns = df.columns if exclude_column is None else [c for c in df.columns if c != exclude_column]

#     # Calculate the percentage of null values for each column
#     null_percentages = df.select(
#         *[(count(when(col(c).isNull(), c)) / count(lit(1))).alias(c) for c in columns]
#     ).collect()[0]
    
#     # Convert to a DataFrame for easier manipulation and display
#     null_percentages_list = [(k, v) for k, v in null_percentages.asDict().items()]
#     null_percentages_df = spark.createDataFrame(
#         null_percentages_list,
#         ['column', 'null_percentage']
#     )
    
#     # Filter columns with more than the threshold percentage of null values
#     high_null_columns_df = null_percentages_df.filter(col('null_percentage') > threshold)
    
#     # Count the number of columns with high null percentage
#     high_null_count = high_null_columns_df.count()
    
#     return high_null_columns_df, high_null_count

from pyspark.sql.functions import col, count, when, lit
from functools import reduce

# Define the function to calculate the percentage of null values
def find_columns_with_high_null_percentage(df, threshold=0.9):
    columns = df.columns

    # Calculate the percentage of null values for each column
    null_percentages = df.select(
        *[(count(when(col(c).isNull(), c)) / count(lit(1))).alias(c) for c in columns]
    ).collect()[0]

    # Convert to a DataFrame for easier manipulation and display
    null_percentages_list = [(k, v) for k, v in null_percentages.asDict().items()]
    null_percentages_df = spark.createDataFrame(
        null_percentages_list,
        ['column', 'null_percentage']
    )

    # Filter columns with more than the threshold percentage of null values
    high_null_columns_df = null_percentages_df.filter(col('null_percentage') > threshold)

    # Count the number of columns with high null percentage
    high_null_count = high_null_columns_df.count()

    return high_null_columns_df, high_null_count

In [31]:
#####################Alternate Solution-S2 ###########################################################
# Split the DataFrame
split_dfs = split_df(Lab1, 15)

In [32]:
# high_null_count_total = 0
# updated_splits = []
# # Apply the function to the DataFrame
# for split_df in split_dfs:
#     high_null_columns_df, high_null_count = find_columns_with_high_null_percentage(split_df)
#     high_null_count_total += high_null_count
#     updated_splits.append(split_df)

# # # Show the result
# # high_null_columns_df.show()
# print(f"Total number of columns with more than 90% null values: {high_null_count}")
high_null_count_total = 0

# Apply the function to the DataFrame
for split_df in split_dfs:
    try:
        high_null_columns_df, high_null_count = find_columns_with_high_null_percentage(split_df)
        high_null_count_total += high_null_count
    except Exception as e:
        print(f"Error processing split: {e}")

# Show the result
print(f"Total number of columns with more than 90% null values: {high_null_count_total}")

Total number of columns with more than 90% null values: 4627


In [10]:
high_null_count_total = 0
updated_splits = []

for split_df in split_dfs:
    result_df, high_null_count = find_columns_with_high_null_percentage(split_df, threshold=0.9)
    high_null_count_total += high_null_count
    updated_splits.append(split_df)

print(f"Total number of columns with more than 90% null values: {high_null_count_total}")


Total number of columns with more than 90% null values: 4627


In [11]:
# Join updated splits
joined_df = updated_splits[0]
for i in range(1, len(updated_splits)):
    print(f"Joining DataFrame {i+1}...")
    joined_df = joined_df.join(updated_splits[i], on='personid', how='inner')

Joining DataFrame 2...
Joining DataFrame 3...
Joining DataFrame 4...
Joining DataFrame 5...
Joining DataFrame 6...
Joining DataFrame 7...
Joining DataFrame 8...
Joining DataFrame 9...
Joining DataFrame 10...
Joining DataFrame 11...
Joining DataFrame 12...
Joining DataFrame 13...
Joining DataFrame 14...
Joining DataFrame 15...


In [12]:
joined_df.show(truncate=False)

+------------------------------------+-------+-------+-------+-------+-------+-------+-------+-------+------+--------+-------+-------+-------+------+-------+-------+-------+------+------+-------+-------+------+-------+-------+-------+------+-------+------+-------+-------+-------+------+------+-------+-------+-------+-------+-------+-------+-------+-----+-------+------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+------+-------+-------+-------+------+-------+------+-------+------+-------+-------+-------+------+-------+------+-------+------+-------+-------+-------+------+-------+-------+------+-------+------+-------+-------+-------+------+-----+-------+-------+-------+------+-------+-------+-------+------+-------+-------+-------+------+-------+-------+-------+------+------+-------+------+-------+-------+-------+------+-------+-------+-------+-------+-------+-------+------+------+-------+------+-------+-------+-------+------+------+------+-------+-------

In [13]:
joined_df.count()

118524

In [ ]:
# Write the updated DataFrame to parquet
output_path = 'file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Cohort-nullcount'
try:
    joined_df.write.mode('overwrite').parquet(output_path)
    print(f"DataFrame successfully written to {output_path}")
except Exception as e:
    print(f"Error writing DataFrame: {e}")

In [10]:
# Join updated splits
joined_df = updated_splits[0]
for i in range(1, len(updated_splits)):
    print(f"Joining DataFrame {i+1}...")
    joined_df = joined_df.join(updated_splits[i], on='personid', how='inner')

▸,:,


Joining DataFrame 2...
Joining DataFrame 3...
Joining DataFrame 4...
Joining DataFrame 5...
Joining DataFrame 6...
Joining DataFrame 7...
Joining DataFrame 8...
Joining DataFrame 9...
Joining DataFrame 10...
Joining DataFrame 11...
Joining DataFrame 12...
Joining DataFrame 13...
Joining DataFrame 14...
Joining DataFrame 15...


In [11]:
#####################Alternate Solution-S5 ###########################################################
# Show the final joined DataFrame
joined_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|personid                            |null_percentage   |null_percentage   |null_percentage   |null_percentage   |null_percentage   |null_percentage   |null_percentage   |null_percentage   |null_percentage   |null_percentage   |null_percentage   |null_percentage   |null_percentage   |null_percentage   |null_percentage   |
+------------------------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|0053f0b8-4a78-4826-b193-9a9

<IPython.core.display.Javascript object>

In [12]:
print(joined_df.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

118524


<IPython.core.display.Javascript object>

In [4]:
joined_df.printSchema()

▸,:,


NameError: name 'joined_df' is not defined